In [0]:
gold_bt = spark.table("workspace.default.silver_bank_transactions")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import countDistinct, count_if, col, stddev, avg, variance, corr, percentile_approx, max, first, last, collect_list, collect_set, sum, round, row_number, rank, dense_rank, lag, lead, sum as spark_sum

aggregation

In [0]:
gold_bt.agg(countDistinct("CustLocation")).show()

In [0]:
gold_bt.printSchema()

In [0]:
gold_bt.agg(countDistinct("CustomerID")).show()

In [0]:
gold_bt.agg(
    countDistinct("CustomerID").alias("DistinctCustomers"),
    countDistinct("CustLocation").alias("DistinctLocations")
).show()

In [0]:
gold_bt.agg(count_if(col("TransactionAmountINR") > 50000)).show()

In [0]:
gold_bt.agg(count_if(col("TransactionAmountINR") < 1000).alias("LowBalanceTxns")).show()

Stats

measures how spread out values are around the average. A low stddev means most values cluster close to the mean; a high one means they're scattered widely.

In [0]:
gold_bt.agg(
    avg(col("TransactionAmountINR")).alias("AvgTxnAmt"),
    stddev(col("TransactionAmountINR"))
).show()

In [0]:
gold_bt.agg(variance(col("TransactionAmountINR"))).show()

corr() only works on numeric columns, because correlation is fundamentally a mathematical relationship between quantities

In [0]:
gold_bt.agg(corr(col("CustAccountBalance"), col("TransactionAmountINR"))).show()

position-based functions — first(), last(), percentile_approx().

In [0]:
gold_bt.agg(percentile_approx(col("TransactionAmountINR"), 0.5)).show()

In [0]:
gold_bt.agg(percentile_approx(col("TransactionAmountINR"), [0.25, 0.5, 0.75])).show()

first() and last().

What they do: within a group (or across a whole DataFrame), grab the first or last value encountered — not the biggest, not the smallest, just whichever row happens to come first or last in whatever order Spark processes the data.

The important warning, worth internalizing before you ever use these for real: unless you explicitly sort the data first, "first" and "last" have no reliable meaning. Spark processes data across a distributed cluster, in whatever order is most efficient — there's no guaranteed "natural order" to a table the way there might be in a spreadsheet. So first() without an explicit orderBy() beforehand can give you a different row every time you run the exact same query.

In [0]:
gold_bt.groupBy("CustLocation").agg(max(col("TransactionDate")).alias("MostRecentTxn")).show(10)

explicitly sort the data by date (most recent first) before grouping, else first() will give different answers each time

In [0]:
gold_bt.orderBy(col("TransactionDate")).groupBy("CustLocation").agg(first(col("TransactionDate"))).show()

max()/min() are the correct whenever you genuinely want the largest/smallest value — they're mathematically defined and don't depend on row order at all. first()/last() should really only be used when you deliberately want "whichever row happens to represent this group"

collect_list keeps every value, including duplicates. collect_set removes duplicates.

In [0]:
# for each CustLocation, collect the set of unique CustGender
gold_bt.groupBy("CustLocation").agg(collect_set("CustGender")).alias("Genderbyloc").show(20)

In [0]:
# remember to .filter(...) down to a single city before the groupBy/agg, so you're not trying to collect a million-row array
gold_bt.groupBy("CustLocation").agg(collect_list("CustGender").alias("Genderbyloc")).show(10)
# alias inside agg
# gold_bt.groupBy("CustLocation").agg(collect_list("CustGender").alias("GendersByLoc")).show(10, truncate=False)

In [0]:
gold_bt.filter(col("CustLocation") == "GODDA").groupBy("CustLocation").agg(collect_list(col("TransactionAmountINR")).alias("AllAmounts")).show(truncate=False)

In [0]:
gold_bt.filter(col("CustLocation") == "JHAJJAR").groupBy("CustLocation").agg(collect_list(col("TransactionAmountINR")).alias("AllAmounts")).show(truncate=False)

advance agg functions

df.rollup() is an advanced aggregation function used to calculate hierarchical subtotals and grand totals within a single query

The order of columns matters. rollup(A, B) will not calculate a subtotal for B alone.

In [0]:
gold_bt.filter(col("CustLocation") == "GODDA").rollup("CustLocation", "CustGender").agg(sum(col("TransactionAmountINR"))).show()

& -> and
| -> or

In [0]:
gold_bt.filter((col("CustLocation") == "GODDA") | (col("CustLocation") == "JHAJJAR")).rollup("CustLocation", "CustGender").agg(round(sum(col("TransactionAmountINR")), 0).alias("TotalAmount")).show()

# THE NULL NULL IS THE GRANDTOTAL !!!!!!!!!!!!!!!!!!1

Provides all possible combinations regardless of hierarchy

In [0]:
# gold_bt.filter((col("CustLocation")== "GODDA") | (col("CustLocation") == "JHAJJAR")).cube("CustLocation", "CustGender").agg(round(sum(col("TransactionAmountINR")),0).alias("Total")).show()

# output layout fix 
gold_bt.filter((col("CustLocation")== "GODDA") | (col("CustLocation") == "JHAJJAR")).cube("CustLocation", "CustGender").agg(round(sum(col("TransactionAmountINR")),0).alias("Total")).orderBy(col("CustLocation").asc_nulls_last(), col("CustGender").asc_nulls_last()) \
    .show()


.asc_nulls_last() — a specific sort direction with an extra rule for nulls. Break this into two ideas:

asc = ascending order (A→Z, smallest→largest) — the normal default
nulls_last = a specific instruction for where null values land, since null doesn't have a natural "size" to sort by on its own

In [0]:
gold_bt.filter((col("CustLocation") == "GODDA") | (col("CustLocation") == "JHAJJAR")) \
    .groupBy("CustLocation") \
    .pivot("CustGender") \
    .agg(round(sum(col("TransactionAmountINR")), 2).alias("Total")) \
    .show()

In [0]:
gold_bt.filter((col("CustLocation") == "ITANAGAR") | (col("CustLocation") == "MOHALI")) \
    .groupBy("CustLocation") \
    .pivot("CustGender") \
    .agg(round(avg(col("CustAccountBalance")), 0).alias("Total")) \
    .show()

window functions

In [0]:
window_spec = Window.partitionBy("CustLocation")
# just defines the grouping

In [0]:
window_spec = Window.partitionBy("CustLocation").orderBy(col("CustAccountBalance").desc())
# just defines the grouping

In [0]:
small_df = gold_bt.filter((col("CustLocation") == "GODDA") | (col("CustLocation") == "JHAJJAR"))

In [0]:
result_df = small_df.withColumn("BalanceRank", row_number().over(window_spec))

In [0]:
result_df.select("CustLocation", "CustomerID", "CustAccountBalance", "BalanceRank").show(10)

In [0]:
# gold_bt.withColumn("GnederTxn", dense_rank().over(Window.partitionBy("CustGender").orderBy(col("TransactionAmountINR").desc()))).show(20)

# Better query
gold_bt.filter(col("CustGender") == "M") \
    .withColumn("GenderTxnRank", dense_rank().over(Window.partitionBy("CustGender").orderBy(col("TransactionAmountINR").desc()))) \
    .select("CustGender", "CustomerID", "TransactionAmountINR", "GenderTxnRank") \
    .show(10)

In [0]:
gold_bt.filter(col("CustLocation").isNotNull()).withColumn("CityTxn", rank().over(Window.partitionBy("CustLocation").orderBy(col("TransactionDateTime").desc())))\
    .select("TransactionDateTime", "CustLocation", "CityTxn").show(10)

gold_bt
    .filter(...)                                    # step 1: narrow down rows
    .withColumn(                                     # step 2: add new column, containing...
        "SomeRankColumn",
        rank().over(                                  #   ...a window function, applied over...
            Window.partitionBy(...).orderBy(...)      #   ...a window definition (partition + sort)
        )
    )
    .select(...)                                      # step 3: pick columns to display
    .show()                                            # step 4: actually display it

In [0]:
gold_bt.filter((col("CustLocation") == "GODDA") & (col("CustGender") == 'M')).withColumn("PrevTxnAmt", lag(col("TransactionAmountINR")).over(Window.partitionBy("CustLocation").orderBy(col("TransactionDateTime").asc()))) \
    .select("CustLocation", "TransactionDateTime", "TransactionAmountINR", "PrevTxnAmt").show(10)

# sorting asc because we want prev transaction

In [0]:
lead(col("TransactionAmtINR").over(Window.partitionBy("CustGender").orderBy(col("TransactionDateTime").desc())))
# lag/lead are defined purely by position within whatever order you specify — they have no built-in understanding of "past" or "future" on their own. It's the orderBy direction that gives "previous"/"next" their real-world meaning

In [0]:
# gold_bt.select(col("TransactionID"), lag(col("TransactionAmountINR")).over(window_spec).alias("PrevAmt"))

In [0]:
# spark_sum(col("TransactionAmountINR")).over(Window.partitionBy("CustLocation").orderBy(col("TransactionDateTime").asc()))

Topics remaining: \
join() \
MERGE — \
Delta-specific table operations — OPTIMIZE, VACUUM, and time travel (querying a table's state as of an earlier version) \
UDFs (user-defined functions) — writing custom Python logic \